# Teleconnection indices over the study period

ENSO (Niño 3.4, the long ERSST series), PDO (ensemble SST) and PNA monthly indices, read live from NOAA PSL and
shown over the water years of the dataset (context for the regional anomaly maps). Needs only the dataset config
(water years) and an internet connection; the series are not cached, so the figure follows NOAA's updates.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from gsro_analysis import paths, settings

config = settings.load_config()

In [ ]:
# NOAA PSL monthly indices, read live. Missing values are -9999 (and -99.9 / -99.99 in the headers' own conventions).
# Nino 3.4: the long ERSST series; the shorter nina34.anom.csv on the same site stopped in 1994 (checked 2026-09-04).
MISSING = [-9999, -9999.0, -99.9, -99.99]
enso_df = pd.read_csv('https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.csv', skiprows=0, na_values=MISSING)
enso_df.rename(columns={enso_df.columns[1]: 'ENSO'}, inplace=True)
enso_df['time'] = pd.to_datetime(enso_df['Date'])
enso_df = enso_df.set_index('time').drop(columns=['Date'])

pdo_df = pd.read_csv('https://psl.noaa.gov/data/timeseries/month/data/pdo.timeseries.sstens.csv', skiprows=0, na_values=MISSING)
pdo_df.rename(columns={pdo_df.columns[1]: 'PDO'}, inplace=True)
pdo_df['time'] = pd.to_datetime(pdo_df['Date'])
pdo_df = pdo_df.set_index('time').drop(columns=['Date'])


pna_df = pd.read_csv('https://psl.noaa.gov/data/correlation/pna.csv', skiprows=0, na_values=MISSING)
pna_df.rename(columns={pna_df.columns[1]: 'PNA'}, inplace=True)
pna_df['time'] = pd.to_datetime(pna_df['Date'])
pna_df = pna_df.set_index('time').drop(columns=['Date'])

print(f"ENSO {enso_df.index.min().date()}..{enso_df.dropna().index.max().date()} | PDO ..{pdo_df.dropna().index.max().date()} | PNA ..{pna_df.dropna().index.max().date()}")
enso_df.dropna().tail()


In [ ]:
enso_colors = enso_df['ENSO'].apply(lambda x: 'red' if x > 0.5 else ('blue' if x < -0.5 else 'grey'))

pdo_colors = pdo_df['PDO'].apply(lambda x: 'red' if x > 0 else 'blue')

pna_colors = pna_df['PNA'].apply(lambda x: 'red' if x > 0 else 'blue')

# each series is plotted on its own dates; the axes limits below select the water years of the dataset

# Plotting
f, axs = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# ENSO bar plot
axs[0].bar(enso_df.index, enso_df['ENSO'], color=enso_colors, width=20)
axs[0].set_title('ENSO Index (Nino 3.4 ERSSTv5)')
axs[0].set_ylabel('value')
axs[0].axhline(0, color='black', linestyle='--', linewidth=0.5)
axs[0].axhline(0.5, color='black', linestyle='--', linewidth=0.5)
axs[0].axhline(-0.5, color='black', linestyle='--', linewidth=0.5)

# PDO bar plot
axs[1].bar(pdo_df.index, pdo_df['PDO'], color=pdo_colors, width=20)
axs[1].set_title('Pacific Decadal Oscillation (PDO)')
axs[1].set_ylabel('value')
axs[1].axhline(0, color='black', linestyle='--', linewidth=0.5)

# PNA bar plot
axs[2].bar(pna_df.index, pna_df['PNA'], color=pna_colors, width=20)
axs[2].set_title('Pacific North American Index (PNA)')
axs[2].set_ylabel('value')
axs[2].axhline(0, color='black', linestyle='--', linewidth=0.5)


axs[0].set_xlim([pd.to_datetime(f'{config.water_years[0]-1}-10-01'), pd.to_datetime(f'{config.water_years[-1]}-09-30')])


for ax in axs:
    for year in config.water_years[:-1]:
        ax.axvline(pd.to_datetime(f'{year}-10-01'), color='black', linestyle='--', linewidth=0.5)

f.tight_layout()

In [ ]:
# Setup figure and axes
f, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6),dpi=300)

# Plot ENSO data
ax1.bar(enso_df.index, enso_df['ENSO'], color=enso_colors, width=20)
ax1.set_ylabel('ENSO Index')
ax1.yaxis.set_label_position("left")
ax1.yaxis.tick_right()
ax1.axhline(0, color='black', linestyle='--', linewidth=0.5)
#ax1.axhline(0.5, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
#ax1.axhline(-0.5, color='black', linestyle='--', linewidth=0.5, alpha=0.5)

# Plot PDO data
ax2.bar(pdo_df.index, pdo_df['PDO'], color=pdo_colors, width=20)
ax2.set_ylabel('PDO Index')
ax2.yaxis.set_label_position("left")
ax2.yaxis.tick_right()
ax2.axhline(0, color='black', linestyle='--', linewidth=0.5)

# Format x-axis ticks
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax2.tick_params(axis='x', labelrotation=45)

plt.setp(ax2.get_xticklabels(), rotation=90)

ax1.set_xticklabels([])
#ax1.set_frame_on(False)
#ax2.set_frame_on(False)
ax1.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

# Set date limits
start_date = pd.to_datetime(f'{config.water_years[0]-1}-10-01')  # water-year span from config
end_date = pd.to_datetime(f'{config.water_years[-1]}-09-30')
ax1.set_xlim([start_date, end_date])
ax2.set_xlim([start_date, end_date])

# Add water year shading and centered labels
for year in config.water_years:
    wy_start = pd.to_datetime(f'{year-1}-10-01')
    wy_end = pd.to_datetime(f'{year}-09-30')
    mid_date = wy_start + pd.Timedelta(days=182)  # Approximate middle of water year
    
    if year % 2 == 0:
        ax1.axvspan(wy_start, wy_end, color='gray', alpha=0.2)
        ax2.axvspan(wy_start, wy_end, color='gray', alpha=0.2)

    ax1.axvline(wy_start, color='black', linestyle='--', linewidth=0.5)
    ax2.axvline(wy_start, color='black', linestyle='--', linewidth=0.5)
        
    # Winter period shading (Oct-Mar)
    winter_start = pd.to_datetime(f'{year-1}-10-01')
    winter_end = pd.to_datetime(f'{year}-03-31')
    

    
    # Convert timestamp to matplotlib date number for annotation
    mid_date_num = mdates.date2num(mid_date)
    
    # Calculate position in figure coordinates
    display_coords = ax2.transData.transform((mid_date_num, 0))
    fig_coords = f.transFigure.inverted().transform(display_coords)
    
    # Add water year label
    f.text(fig_coords[0], 0.91, f'WY {year}', 
           ha='center', va='top',
           transform=f.transFigure, fontdict={'fontsize': 10})

# Add legends
enso_legend_elements = [
    plt.Rectangle((0,0),1,1, color='red', label='El Niño (>0.5)'),
    plt.Rectangle((0,0),1,1, color='blue', label='La Niña (<-0.5)'),
    plt.Rectangle((0,0),1,1, color='grey', label='Neutral')
]
ax1.legend(handles=enso_legend_elements, loc='lower right', framealpha=1)

pdo_legend_elements = [
    plt.Rectangle((0,0),1,1, color='red', label='Positive Phase'),
    plt.Rectangle((0,0),1,1, color='blue', label='Negative Phase')
]
ax2.legend(handles=pdo_legend_elements, loc='upper right', framealpha=1)


ax1.spines['right'].set_visible(True)
ax2.spines['right'].set_visible(True)
ax1.spines['right'].set_linewidth(1)
ax2.spines['right'].set_linewidth(1)

# Add horizontal line between plots
ax1.spines['bottom'].set_visible(True)
ax1.spines['bottom'].set_linewidth(1)

ax1.spines['top'].set_visible(True)
ax2.spines['top'].set_visible(True)
ax2.spines['bottom'].set_visible(True)

# Layout adjustments
plt.subplots_adjust(hspace=0.00)

f.savefig(paths.figdir('mountain_ranges', config.version) / 'teleconnection_indices.png',dpi=300)